# Notebook 10 -- Feature Integration

<div style="border-left: 4px solid #4680a7; padding: 10px 15px; margin: 10px 0; background: #4e6681;">

**Objective:** Merge all modality-specific feature sets (tabular, text, image) into a single unified feature matrix. Validate cross-modality alignment, register the integrated schema with modality provenance tags, and produce the final train/test datasets ready for model training.

**Answers:** *"Is the integrated multimodal feature matrix complete, consistent, and ready for modeling?"*

</div>

| Item | Detail |
|------|--------|
| **Dependencies** | `data/features/tabular/v1/`, `data/features/text/v1/`, `data/features/images/v1/`, `data/cleaned/train.parquet` |
| **Artifacts** | `data/features/integrated/v1/train.parquet`, `data/features/integrated/v1/test.parquet`, `data/features/integrated/v1/schema.json`, `reports/feature_integration.json` |
| **Scope** | Feature alignment and integration only -- no new feature engineering, no dimensionality reduction, no transformations |
| **Runtime** | < 30 seconds |

---
## 1. Imports & Configuration

In [20]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

from adoption_accelerator import config
from adoption_accelerator.features.integration import (
    audit_integrated_matrix,
    build_provenance_map,
    load_modality_features,
    merge_modality_dataframes,
    validate_petid_alignment,
)
from adoption_accelerator.features.registry import (
    build_column_descriptors,
    compute_config_hash,
    load_feature_schema,
    save_feature_schema,
)

# -- Constants -------------------------------------------------------------
SEED = config.SEED
EXPECTED_TRAIN_ROWS = 14_993
EXPECTED_TEST_ROWS = 3_972

# -- Feature source versions -----------------------------------------------
FEATURE_VERSIONS = {
    "tabular": {"dir": "tabular", "version": "v1"},
    "text":    {"dir": "text",    "version": "v1"},
    "image":   {"dir": "images",  "version": "v1"},
}

TARGET_VERSION = "v1"
APPLY_SCALING = False          # Tree-based models do not require scaling
SCALING_METHOD = "standard"    # Placeholder for future neural-network pipelines

# -- Output paths ----------------------------------------------------------
OUTPUT_DIR = config.DATA_FEATURES / "integrated" / TARGET_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = config.REPORTS_DIR / "feature_integration.json"

t0 = time.time()
print(f"SEED             : {SEED}")
print(f"Target version   : {TARGET_VERSION}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Apply scaling    : {APPLY_SCALING}")
print(f"Modality sources :")
for m, info in FEATURE_VERSIONS.items():
    print(f"  {m:>10s} -> {info['dir']}/{info['version']}")

SEED             : 42
Target version   : v1
Output directory : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\integrated\v1
Apply scaling    : False
Modality sources :
     tabular -> tabular/v1
        text -> text/v1
       image -> images/v1


---
## 2. Load Modality Feature Sets

<div style="border-left: 4px solid #f39c12; padding: 10px 15px; margin: 10px 0; background: #5e5535;">

**Strategy:** Load versioned Parquet outputs and their `schema.json` from notebooks 07 (tabular), 08 (text), and 09 (images). Validate schema structure before proceeding.

</div>

In [2]:
# -- Load all modality features -------------------------------------------
train_dfs: dict[str, pd.DataFrame] = {}
test_dfs: dict[str, pd.DataFrame] = {}
schemas: dict[str, dict] = {}

for modality, info in FEATURE_VERSIONS.items():
    dir_name = info["dir"]
    version = info["version"]

    train_df, schema = load_modality_features(dir_name, version, "train")
    test_df, _ = load_modality_features(dir_name, version, "test")

    # Ensure PetID is the index (not a column)
    if "PetID" in train_df.columns:
        train_df = train_df.set_index("PetID")
    if "PetID" in test_df.columns:
        test_df = test_df.set_index("PetID")

    train_dfs[modality] = train_df
    test_dfs[modality] = test_df
    schemas[modality] = schema

    print(
        f"  {modality:>10s}  |  train: {train_df.shape[0]:>6,} x {train_df.shape[1]:<4}  |  "
        f"test: {test_df.shape[0]:>6,} x {test_df.shape[1]:<4}  |  "
        f"schema v={schema['version']}  hash={schema.get('config_hash', 'N/A')}"
    )

print(f"\nLoaded {len(FEATURE_VERSIONS)} modality feature sets.")

Column mismatch for tabular/v1/train: extra={'PetID'}, missing=set()
Column mismatch for tabular/v1/test: extra={'PetID'}, missing=set()
Column mismatch for text/v1/train: extra={'PetID'}, missing=set()
Column mismatch for text/v1/test: extra={'PetID'}, missing=set()
Column mismatch for images/v1/train: extra={'PetID'}, missing=set()
Column mismatch for images/v1/test: extra={'PetID'}, missing=set()


     tabular  |  train: 14,993 x 45    |  test:  3,972 x 45    |  schema v=v1  hash=4f6dff7ee5aba516
        text  |  train: 14,993 x 784   |  test:  3,972 x 784   |  schema v=v1  hash=3462c7de003cc4d6
       image  |  train: 14,993 x 111   |  test:  3,972 x 111   |  schema v=v1  hash=5e84262639d30746

Loaded 3 modality feature sets.


---
## 3. Validation Gate G10-1 -- Schema & Load Integrity

In [3]:
# -- Validation Gate G10-1: All modality feature files loaded with valid schemas
for modality in FEATURE_VERSIONS:
    assert modality in train_dfs, f"G10-1 FAIL: {modality} train features not loaded"
    assert modality in test_dfs, f"G10-1 FAIL: {modality} test features not loaded"
    assert modality in schemas, f"G10-1 FAIL: {modality} schema not loaded"

    s = schemas[modality]
    assert "version" in s, f"G10-1 FAIL: {modality} schema missing 'version'"
    assert "columns" in s, f"G10-1 FAIL: {modality} schema missing 'columns'"
    assert "n_rows_train" in s, f"G10-1 FAIL: {modality} schema missing 'n_rows_train'"
    assert "n_rows_test" in s, f"G10-1 FAIL: {modality} schema missing 'n_rows_test'"

print("G10-1 PASS: All three modality feature files loaded with valid schemas.")
for m, s in schemas.items():
    print(f"   {m}: v={s['version']}, modality={s['modality']}, n_features={s['n_features']}")

G10-1 PASS: All three modality feature files loaded with valid schemas.
   tabular: v=v1, modality=tabular, n_features=45
   text: v=v1, modality=text, n_features=784
   image: v=v1, modality=image, n_features=111


---
## 4. PetID Alignment Audit

In [4]:
# -- PetID alignment audit ------------------------------------------------
train_alignment = validate_petid_alignment(train_dfs, "train")
test_alignment = validate_petid_alignment(test_dfs, "test")

print("Train PetID alignment:")
for m, count in train_alignment["per_modality_count"].items():
    print(f"   {m:>10s}: {count:,} PetIDs")
print(f"   Aligned: {train_alignment['aligned']}")

print(f"\nTest PetID alignment:")
for m, count in test_alignment["per_modality_count"].items():
    print(f"   {m:>10s}: {count:,} PetIDs")
print(f"   Aligned: {test_alignment['aligned']}")

Train PetID alignment:
      tabular: 14,993 PetIDs
         text: 14,993 PetIDs
        image: 14,993 PetIDs
   Aligned: True

Test PetID alignment:
      tabular: 3,972 PetIDs
         text: 3,972 PetIDs
        image: 3,972 PetIDs
   Aligned: True


In [5]:
# -- Validation Gate G10-2: PetID sets identical across all modalities -----
assert train_alignment["aligned"], (
    f"G10-2 FAIL: Train PetID sets are NOT aligned. "
    f"Mismatches: {train_alignment['mismatches']}"
)
assert test_alignment["aligned"], (
    f"G10-2 FAIL: Test PetID sets are NOT aligned. "
    f"Mismatches: {test_alignment['mismatches']}"
)
print("G10-2 PASS: PetID sets are identical across all modalities for both splits.")
print(f"   Train: {train_alignment['per_modality_count'][list(train_alignment['per_modality_count'].keys())[0]]:,} PetIDs")
print(f"   Test:  {test_alignment['per_modality_count'][list(test_alignment['per_modality_count'].keys())[0]]:,} PetIDs")

G10-2 PASS: PetID sets are identical across all modalities for both splits.
   Train: 14,993 PetIDs
   Test:  3,972 PetIDs


---
## 5. Merge Modality DataFrames

In [6]:
# -- Merge features (horizontal concatenation on PetID index) -------------
train_merged = merge_modality_dataframes(train_dfs)
test_merged = merge_modality_dataframes(test_dfs)

print(f"Merged train: {train_merged.shape[0]:,} rows x {train_merged.shape[1]:,} columns")
print(f"Merged test:  {test_merged.shape[0]:,} rows x {test_merged.shape[1]:,} columns")

# -- Verify column consistency between splits
train_cols = set(train_merged.columns)
test_cols = set(test_merged.columns)
assert train_cols == test_cols, (
    f"Column mismatch: train-only={train_cols - test_cols}, test-only={test_cols - train_cols}"
)
print(f"Column sets match across splits.")

Merged train: 14,993 rows x 940 columns
Merged test:  3,972 rows x 940 columns
Column sets match across splits.


---
## 6. Missing Modality Audit

In [7]:
# -- Missing modality audit (NaN introduced by outer join) ----------------
train_nan_counts = train_merged.isna().sum()
test_nan_counts = test_merged.isna().sum()

train_cols_with_nan = train_nan_counts[train_nan_counts > 0]
test_cols_with_nan = test_nan_counts[test_nan_counts > 0]

if len(train_cols_with_nan) == 0 and len(test_cols_with_nan) == 0:
    print("No NaN values detected after merge -- all modalities fully aligned.")
else:
    if len(train_cols_with_nan) > 0:
        print(f"WARNING: {len(train_cols_with_nan)} train columns have NaN:")
        for col, cnt in train_cols_with_nan.items():
            print(f"   {col}: {cnt:,} NaN")
    if len(test_cols_with_nan) > 0:
        print(f"WARNING: {len(test_cols_with_nan)} test columns have NaN:")
        for col, cnt in test_cols_with_nan.items():
            print(f"   {col}: {cnt:,} NaN")

No NaN values detected after merge -- all modalities fully aligned.


---
## 7. Attach Target Variable

In [8]:
# -- Attach AdoptionSpeed from cleaned train data -------------------------
cleaned_train = pd.read_parquet(config.DATA_CLEANED / "train.parquet")
if "PetID" in cleaned_train.columns:
    cleaned_train = cleaned_train.set_index("PetID")

target = cleaned_train["AdoptionSpeed"]

# Align target with merged train index
target_aligned = target.reindex(train_merged.index)
assert target_aligned.isna().sum() == 0, (
    f"Target alignment failure: {target_aligned.isna().sum()} missing AdoptionSpeed values"
)

train_merged["AdoptionSpeed"] = target_aligned.values

print(f"AdoptionSpeed attached to train ({train_merged['AdoptionSpeed'].nunique()} classes).")
print(f"Distribution:")
dist = train_merged["AdoptionSpeed"].value_counts().sort_index()
for cls, cnt in dist.items():
    print(f"   Class {cls}: {cnt:>5,} ({cnt / len(train_merged) * 100:.1f}%)")

AdoptionSpeed attached to train (5 classes).
Distribution:
   Class 0:   410 (2.7%)
   Class 1: 3,090 (20.6%)
   Class 2: 4,037 (26.9%)
   Class 3: 3,259 (21.7%)
   Class 4: 4,197 (28.0%)


---
## 8. Column Ordering & Provenance Map

In [9]:
# -- Build provenance map (column -> source modality) ----------------------
provenance_map = build_provenance_map(schemas)

# Tag the target column
provenance_map["AdoptionSpeed"] = "target"

# -- Column ordering: tabular -> text -> image -> target -------------------
tabular_cols = [c["name"] for c in schemas["tabular"]["columns"]]
text_cols = [c["name"] for c in schemas["text"]["columns"]]
image_cols = [c["name"] for c in schemas["image"]["columns"]]

ordered_feature_cols = tabular_cols + text_cols + image_cols
ordered_train_cols = ordered_feature_cols + ["AdoptionSpeed"]
ordered_test_cols = ordered_feature_cols

# Verify all expected columns are present
missing_in_train = set(ordered_train_cols) - set(train_merged.columns)
missing_in_test = set(ordered_test_cols) - set(test_merged.columns)
assert not missing_in_train, f"Missing columns in train: {missing_in_train}"
assert not missing_in_test, f"Missing columns in test: {missing_in_test}"

train_merged = train_merged[ordered_train_cols]
test_merged = test_merged[ordered_test_cols]

print(f"Column ordering applied: tabular({len(tabular_cols)}) -> text({len(text_cols)}) -> image({len(image_cols)})")
print(f"Train columns: {len(train_merged.columns)} (incl. AdoptionSpeed)")
print(f"Test columns:  {len(test_merged.columns)}")

# -- Provenance summary
from collections import Counter
prov_counts = Counter(provenance_map.values())
print(f"\nProvenance map summary:")
for source, count in sorted(prov_counts.items()):
    print(f"   {source:>10s}: {count:>4} columns")

Column ordering applied: tabular(45) -> text(784) -> image(111)
Train columns: 941 (incl. AdoptionSpeed)
Test columns:  940

Provenance map summary:
        image:  111 columns
      tabular:   45 columns
       target:    1 columns
         text:  784 columns


---
## 9. Feature Dimensionality Summary

In [10]:
# -- Dimensionality & memory summary --------------------------------------
n_tab = len(tabular_cols)
n_txt = len(text_cols)
n_img = len(image_cols)
n_total = n_tab + n_txt + n_img

train_mem_mb = train_merged.memory_usage(deep=True).sum() / (1024 ** 2)
test_mem_mb = test_merged.memory_usage(deep=True).sum() / (1024 ** 2)

print("Feature Dimensionality Breakdown")
print("=" * 50)
print(f"  Tabular features : {n_tab:>6}  ({n_tab / n_total * 100:5.1f}%)")
print(f"  Text features    : {n_txt:>6}  ({n_txt / n_total * 100:5.1f}%)")
print(f"  Image features   : {n_img:>6}  ({n_img / n_total * 100:5.1f}%)")
print(f"  {'':->50s}")
print(f"  Total features   : {n_total:>6}")
print()
print("Memory Footprint")
print("=" * 50)
print(f"  Train : {train_mem_mb:>8.2f} MB  ({train_merged.shape[0]:,} rows)")
print(f"  Test  : {test_mem_mb:>8.2f} MB  ({test_merged.shape[0]:,} rows)")
print(f"  Total : {train_mem_mb + test_mem_mb:>8.2f} MB")

Feature Dimensionality Breakdown
  Tabular features :     45  (  4.8%)
  Text features    :    784  ( 83.4%)
  Image features   :    111  ( 11.8%)
  --------------------------------------------------
  Total features   :    940

Memory Footprint
  Train :    54.36 MB  (14,993 rows)
  Test  :    14.39 MB  (3,972 rows)
  Total :    68.76 MB


---
## 10. Persist Integrated Features

In [11]:
# -- Save integrated Parquet files ----------------------------------------
train_path = OUTPUT_DIR / "train.parquet"
test_path = OUTPUT_DIR / "test.parquet"

train_merged.to_parquet(train_path)
test_merged.to_parquet(test_path)

print(f"Saved: {train_path}")
print(f"       {train_merged.shape[0]:,} rows x {train_merged.shape[1]} cols")
print(f"       size: {train_path.stat().st_size / (1024**2):.2f} MB")
print()
print(f"Saved: {test_path}")
print(f"       {test_merged.shape[0]:,} rows x {test_merged.shape[1]} cols")
print(f"       size: {test_path.stat().st_size / (1024**2):.2f} MB")

Saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\integrated\v1\train.parquet
       14,993 rows x 941 cols
       size: 70.04 MB

Saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\integrated\v1\test.parquet
       3,972 rows x 940 cols
       size: 18.11 MB


In [12]:
# -- Generate and save integrated schema.json -----------------------------
all_column_descriptors = []
for col in ordered_feature_cols:
    source = provenance_map.get(col, "unknown")
    dtype_str = str(train_merged[col].dtype)
    all_column_descriptors.append({
        "name": col,
        "dtype": dtype_str,
        "source": source,
        "description": "",
    })

# Build integration config for hash
integration_config = {
    "source_versions": {
        m: {"dir": info["dir"], "version": info["version"], "config_hash": schemas[m].get("config_hash", "")}
        for m, info in FEATURE_VERSIONS.items()
    },
    "apply_scaling": APPLY_SCALING,
    "scaling_method": SCALING_METHOD if APPLY_SCALING else None,
}

schema_metadata = {
    "version": TARGET_VERSION,
    "modality": "integrated",
    "config_hash": compute_config_hash(integration_config),
    "n_rows_train": len(train_merged),
    "n_rows_test": len(test_merged),
    "n_features": n_total,
    "seed": SEED,
    "notes": (
        f"Integrated multimodal feature matrix. "
        f"Tabular({n_tab}) + Text({n_txt}) + Image({n_img}) = {n_total} features. "
        f"No cross-modality scaling applied."
    ),
}

schema_path = save_feature_schema(
    all_column_descriptors, schema_metadata, OUTPUT_DIR / "schema.json"
)

# Append provenance map and source version metadata to schema
with open(schema_path, "r", encoding="utf-8") as f:
    schema_data = json.load(f)

schema_data["provenance_map"] = provenance_map
schema_data["source_versions"] = integration_config["source_versions"]
schema_data["modality_breakdown"] = {
    "tabular": n_tab,
    "text": n_txt,
    "image": n_img,
}

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema_data, f, indent=2, ensure_ascii=False)

print(f"Saved: {schema_path}")
print(f"   provenance_map entries: {len(provenance_map)}")
print(f"   config_hash: {schema_metadata['config_hash']}")

Saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\integrated\v1\schema.json
   provenance_map entries: 941
   config_hash: e8e2c15e2d75fa02


---
## 11. Validation Gates

In [13]:
# -- Comprehensive validation gates ----------------------------------------
gate_results: list[tuple[str, str, str]] = []  # (gate_id, status, message)

# G10-3: No NaN values in any feature column
feature_cols = [c for c in train_merged.columns if c != "AdoptionSpeed"]
train_nan = int(train_merged[feature_cols].isna().sum().sum())
test_nan = int(test_merged.isna().sum().sum())
if train_nan == 0 and test_nan == 0:
    gate_results.append(("G10-3", "PASS", f"No NaN in feature columns (train={train_nan}, test={test_nan})"))
else:
    gate_results.append(("G10-3", "FAIL", f"NaN detected: train={train_nan}, test={test_nan}"))

# G10-4: No infinite values
numeric_train = train_merged[feature_cols].select_dtypes(include=[np.number])
numeric_test = test_merged.select_dtypes(include=[np.number])
train_inf = int(np.isinf(numeric_train).sum().sum())
test_inf = int(np.isinf(numeric_test).sum().sum())
if train_inf == 0 and test_inf == 0:
    gate_results.append(("G10-4", "PASS", f"No infinite values (train={train_inf}, test={test_inf})"))
else:
    gate_results.append(("G10-4", "FAIL", f"Inf detected: train={train_inf}, test={test_inf}"))

# G10-5: No duplicate PetIDs
train_dups = int(train_merged.index.duplicated().sum())
test_dups = int(test_merged.index.duplicated().sum())
if train_dups == 0 and test_dups == 0:
    gate_results.append(("G10-5", "PASS", f"No duplicate PetIDs (train={train_dups}, test={test_dups})"))
else:
    gate_results.append(("G10-5", "FAIL", f"Duplicates: train={train_dups}, test={test_dups}"))

# G10-6: Row counts
if len(train_merged) == EXPECTED_TRAIN_ROWS and len(test_merged) == EXPECTED_TEST_ROWS:
    gate_results.append(("G10-6", "PASS", f"Row counts correct: train={len(train_merged):,}, test={len(test_merged):,}"))
else:
    gate_results.append(("G10-6", "FAIL", f"Row counts: train={len(train_merged):,} (exp {EXPECTED_TRAIN_ROWS:,}), test={len(test_merged):,} (exp {EXPECTED_TEST_ROWS:,})"))

# G10-7: Feature count = sum of modality features
expected_feature_count = n_tab + n_txt + n_img
actual_feature_count = len(feature_cols)
if actual_feature_count == expected_feature_count:
    gate_results.append(("G10-7", "PASS", f"Feature count: {actual_feature_count} = {n_tab}+{n_txt}+{n_img}"))
else:
    gate_results.append(("G10-7", "FAIL", f"Feature count: {actual_feature_count} != {expected_feature_count} ({n_tab}+{n_txt}+{n_img})"))

# G10-8: AdoptionSpeed present in train, absent in test
has_target_train = "AdoptionSpeed" in train_merged.columns
has_target_test = "AdoptionSpeed" in test_merged.columns
if has_target_train and not has_target_test:
    gate_results.append(("G10-8", "PASS", "AdoptionSpeed in train only"))
else:
    gate_results.append(("G10-8", "FAIL", f"Target: train={has_target_train}, test={has_target_test}"))

# G10-9: AdoptionSpeed distribution unchanged
original_dist = cleaned_train["AdoptionSpeed"].value_counts().sort_index()
merged_dist = train_merged["AdoptionSpeed"].value_counts().sort_index()
if original_dist.equals(merged_dist):
    gate_results.append(("G10-9", "PASS", "AdoptionSpeed distribution matches cleaned data"))
else:
    gate_results.append(("G10-9", "FAIL", "AdoptionSpeed distribution changed after merge"))

# Print results
print("Validation Gates")
print("=" * 70)
for gid, status, msg in gate_results:
    icon = "PASS" if status == "PASS" else "FAIL"
    print(f"  [{icon}] {gid}: {msg}")

# Assert all critical gates passed
critical_failures = [(g, m) for g, s, m in gate_results if s == "FAIL"]
assert not critical_failures, (
    f"Critical validation failures: {critical_failures}"
)

Validation Gates
  [PASS] G10-3: No NaN in feature columns (train=0, test=0)
  [PASS] G10-4: No infinite values (train=0, test=0)
  [PASS] G10-5: No duplicate PetIDs (train=0, test=0)
  [PASS] G10-6: Row counts correct: train=14,993, test=3,972
  [PASS] G10-7: Feature count: 940 = 45+784+111
  [PASS] G10-8: AdoptionSpeed in train only
  [PASS] G10-9: AdoptionSpeed distribution matches cleaned data


---
## 12. Parquet Round-Trip Validation

In [14]:
# -- Validation Gate G10-10: Parquet files exist and are loadable ----------
assert train_path.exists(), f"G10-10 FAIL: {train_path} does not exist"
assert test_path.exists(), f"G10-10 FAIL: {test_path} does not exist"
print("G10-10 PASS: train.parquet and test.parquet exist.")

# -- Validation Gate G10-11: schema.json valid with provenance map ---------
loaded_schema = load_feature_schema(OUTPUT_DIR / "schema.json")
assert loaded_schema["modality"] == "integrated", "G10-11 FAIL: schema modality != 'integrated'"

with open(OUTPUT_DIR / "schema.json", "r", encoding="utf-8") as f:
    full_schema = json.load(f)
assert "provenance_map" in full_schema, "G10-11 FAIL: provenance_map missing from schema"
assert "source_versions" in full_schema, "G10-11 FAIL: source_versions missing from schema"
print("G10-11 PASS: schema.json is valid and contains provenance map.")

# -- Validation Gate G10-12: Parquet round-trip integrity -------------------
reload_train = pd.read_parquet(train_path)
reload_test = pd.read_parquet(test_path)

assert reload_train.shape == train_merged.shape, (
    f"G10-12 FAIL: train shape mismatch: {reload_train.shape} vs {train_merged.shape}"
)
assert reload_test.shape == test_merged.shape, (
    f"G10-12 FAIL: test shape mismatch: {reload_test.shape} vs {test_merged.shape}"
)
assert list(reload_train.columns) == list(train_merged.columns), "G10-12 FAIL: train column order changed"
assert list(reload_test.columns) == list(test_merged.columns), "G10-12 FAIL: test column order changed"

# Dtype check
for col in train_merged.columns:
    assert reload_train[col].dtype == train_merged[col].dtype, (
        f"G10-12 FAIL: dtype mismatch for '{col}': {reload_train[col].dtype} vs {train_merged[col].dtype}"
    )

# Sample value check (first 5 rows)
pd.testing.assert_frame_equal(
    reload_train.head(), train_merged.head(),
    check_exact=False, atol=1e-6,
)
pd.testing.assert_frame_equal(
    reload_test.head(), test_merged.head(),
    check_exact=False, atol=1e-6,
)
print("G10-12 PASS: Parquet round-trip integrity confirmed (shape, dtypes, values).")

G10-10 PASS: train.parquet and test.parquet exist.
G10-11 PASS: schema.json is valid and contains provenance map.
G10-12 PASS: Parquet round-trip integrity confirmed (shape, dtypes, values).


In [15]:
# -- Validation Gate G10-13: Memory footprint < 2 GB per split -------------
if train_mem_mb < 2048 and test_mem_mb < 2048:
    print(f"G10-13 PASS: Memory under 2 GB per split (train={train_mem_mb:.1f} MB, test={test_mem_mb:.1f} MB).")
else:
    print(f"G10-13 WARNING: Memory exceeds 2 GB (train={train_mem_mb:.1f} MB, test={test_mem_mb:.1f} MB).")

G10-13 PASS: Memory under 2 GB per split (train=54.4 MB, test=14.4 MB).


---
## 13. Data Quality Audit

In [16]:
# -- Final data quality audit using src/ function -------------------------
audit_train = audit_integrated_matrix(
    train_merged, expected_rows=EXPECTED_TRAIN_ROWS, expected_cols=n_total, split="train"
)
audit_test = audit_integrated_matrix(
    test_merged, expected_rows=EXPECTED_TEST_ROWS, expected_cols=n_total, split="test"
)

print("Data Quality Audit -- Train")
print("=" * 50)
for check_name, result in audit_train.items():
    if isinstance(result, dict):
        status = "PASS" if result.get("pass", True) else "FAIL"
        print(f"  [{status}] {check_name}: {result}")
    else:
        print(f"  {check_name}: {result}")

print(f"\nData Quality Audit -- Test")
print("=" * 50)
for check_name, result in audit_test.items():
    if isinstance(result, dict):
        status = "PASS" if result.get("pass", True) else "FAIL"
        print(f"  [{status}] {check_name}: {result}")
    else:
        print(f"  {check_name}: {result}")

assert audit_train["all_pass"], "Train audit failed -- see details above."
assert audit_test["all_pass"], "Test audit failed -- see details above."

Data Quality Audit -- Train
  [PASS] row_count: {'expected': 14993, 'actual': 14993, 'pass': True}
  [PASS] feature_count: {'expected': 940, 'actual': 940, 'pass': True}
  [PASS] no_nans: {'nan_count': 0, 'pass': True}
  [PASS] no_infs: {'inf_count': 0, 'pass': True}
  [PASS] no_duplicate_petids: {'duplicate_count': 0, 'pass': True}
  [PASS] memory_mb: {'value': np.float64(54.87), 'under_2gb': np.True_}
  all_pass: True

Data Quality Audit -- Test
  [PASS] row_count: {'expected': 3972, 'actual': 3972, 'pass': True}
  [PASS] feature_count: {'expected': 940, 'actual': 940, 'pass': True}
  [PASS] no_nans: {'nan_count': 0, 'pass': True}
  [PASS] no_infs: {'inf_count': 0, 'pass': True}
  [PASS] no_duplicate_petids: {'duplicate_count': 0, 'pass': True}
  [PASS] memory_mb: {'value': np.float64(14.52), 'under_2gb': np.True_}
  all_pass: True


---
## 14. Distribution Sanity Checks

In [17]:
# -- High-level distribution sanity checks --------------------------------
print("Numeric Feature Statistics (train, sample of key features)")
print("=" * 70)

sample_features = [
    # Tabular
    "is_dog", "Age", "Fee", "health_care_score", "PhotoAmt",
    # Text
    "description_length", "word_count", "doc_sentiment_score",
    # Image
    "has_image_embedding", "actual_photo_count", "mean_image_brightness",
]
sample_features = [f for f in sample_features if f in train_merged.columns]

stats = train_merged[sample_features].describe().T[["mean", "std", "min", "50%", "max"]]
stats.columns = ["mean", "std", "min", "median", "max"]
print(stats.to_string())

# Embedding range check
emb_cols_text = [c for c in train_merged.columns if c.startswith("text_emb_")]
emb_cols_img = [c for c in train_merged.columns if c.startswith("img_emb_")]

print(f"\nEmbedding Value Ranges:")
if emb_cols_text:
    txt_min = train_merged[emb_cols_text].min().min()
    txt_max = train_merged[emb_cols_text].max().max()
    txt_mean = train_merged[emb_cols_text].mean().mean()
    print(f"  Text  ({len(emb_cols_text)} dims): min={txt_min:.4f}, max={txt_max:.4f}, mean={txt_mean:.4f}")

if emb_cols_img:
    img_min = train_merged[emb_cols_img].min().min()
    img_max = train_merged[emb_cols_img].max().max()
    img_mean = train_merged[emb_cols_img].mean().mean()
    print(f"  Image ({len(emb_cols_img)} dims): min={img_min:.4f}, max={img_max:.4f}, mean={img_mean:.4f}")

Numeric Feature Statistics (train, sample of key features)
                             mean         std  min  median      max
is_dog                   0.542386    0.498217  0.0     1.0     1.00
Age                     10.452078   18.155790  0.0     3.0   255.00
Fee                     21.259988   78.414548  0.0     0.0  3000.00
health_care_score        1.160275    1.123743  0.0     1.0     3.00
PhotoAmt                 3.889215    3.487810  0.0     3.0    30.00
description_length     339.221437  373.337962  1.0   238.0  6664.00
word_count              62.952178   69.296581  1.0    44.0  1257.00
doc_sentiment_score      0.270626    0.276728 -0.9     0.2     0.90
has_image_embedding      0.977256    0.149091  0.0     1.0     1.00
actual_photo_count       3.889215    3.487810  0.0     3.0    30.00
mean_image_brightness  117.107269   29.033333  0.0   119.3   229.23

Embedding Value Ranges:
  Text  (768 dims): min=-0.2364, max=0.2395, mean=-0.0001
  Image (100 dims): min=-6.1723, max=6.917

---
## 15. Integration Report

In [18]:
# -- Build and save integration report ------------------------------------
elapsed = time.time() - t0

report = {
    "target_version": TARGET_VERSION,
    "source_versions": {
        m: {
            "dir": info["dir"],
            "version": info["version"],
            "config_hash": schemas[m].get("config_hash", ""),
            "model_name": schemas[m].get("model_name"),
            "n_features": schemas[m].get("n_features"),
        }
        for m, info in FEATURE_VERSIONS.items()
    },
    "integration_config": {
        "apply_scaling": APPLY_SCALING,
        "scaling_method": SCALING_METHOD if APPLY_SCALING else None,
        "config_hash": schema_metadata["config_hash"],
    },
    "dimensions": {
        "n_features_total": n_total,
        "n_tabular": n_tab,
        "n_text": n_txt,
        "n_image": n_img,
    },
    "row_counts": {
        "train": len(train_merged),
        "test": len(test_merged),
    },
    "memory_mb": {
        "train": round(train_mem_mb, 2),
        "test": round(test_mem_mb, 2),
    },
    "alignment": {
        "train": train_alignment,
        "test": test_alignment,
    },
    "audit": {
        "train": audit_train,
        "test": audit_test,
    },
    "validation_gates": [
        {"gate": g, "status": s, "message": m}
        for g, s, m in gate_results
    ],
    "elapsed_seconds": round(elapsed, 2),
    "seed": SEED,
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"Integration report saved: {REPORT_PATH}")
print(f"Total elapsed: {elapsed:.1f}s")

Integration report saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_integration.json
Total elapsed: 65.7s


---
## 16. Final Integrated Matrix Preview

In [19]:
# -- Preview ---------------------------------------------------------------
print(f"Integrated train shape: {train_merged.shape}")
print(f"Integrated test shape:  {test_merged.shape}")
print(f"Index name: {train_merged.index.name}")
print(f"\nFirst 5 columns: {list(train_merged.columns[:5])}")
print(f"Last 5 columns:  {list(train_merged.columns[-5:])}")
print()
train_merged.head(3)

Integrated train shape: (14993, 941)
Integrated test shape:  (3972, 940)
Index name: PetID

First 5 columns: ['is_dog', 'Gender', 'MaturitySize', 'FurLength', 'Health']
Last 5 columns:  ['meta_dominant_color_count_mean', 'meta_avg_brightness_mean', 'meta_color_diversity_mean', 'meta_crop_confidence_mean', 'AdoptionSpeed']



,is_dog,Gender,MaturitySize,FurLength,Health,Vaccinated,Dewormed,Sterilized,health_care_score,Age,...,mean_image_brightness,mean_blur_score,image_size_std,meta_label_count_mean,meta_top_label_score_mean,meta_dominant_color_count_mean,meta_avg_brightness_mean,meta_color_diversity_mean,meta_crop_confidence_mean,AdoptionSpeed
PetID,,,,,,,,,,,,,,,,,,,,,
86e1089a3,0,1,1,1,1,0,0,0,0,3,...,95.86,669.55,0.0,9.0000,0.9908,10.0,124.900002,70.389999,0.8,2
6296e909a,0,1,2,2,1,-1,-1,-1,0,1,...,107.68,330.21,15211.5,8.5000,0.9836,10.0,77.449997,58.560001,0.8,0
3422e4906,1,1,2,2,1,1,1,0,2,1,...,146.63,396.91,0.0,9.7143,0.9657,10.0,119.510002,43.150002,0.8,3


---
## Summary

<div style="border-left: 4px solid #27ae60; padding: 10px 15px; margin: 10px 0; background: #232a3b;">

**Feature Integration -- Complete**

1. **Loaded** -- Tabular (45), Text (784), and Image (111) feature sets from versioned Parquet stores.
2. **Validated** -- PetID alignment confirmed across all three modalities for both train and test splits.
3. **Merged** -- Horizontal concatenation on PetID index producing 940 total features.
4. **Target attached** -- AdoptionSpeed column joined to train split only.
5. **Schema registered** -- `schema.json` with provenance map, source version traceability, and modality breakdown.
6. **Audited** -- All 13 validation gates passed: no NaN, no Inf, no duplicates, correct row counts, round-trip integrity.

**Artifacts produced:**
- `data/features/integrated/v1/train.parquet` (14,993 x 941)
- `data/features/integrated/v1/test.parquet` (3,972 x 940)
- `data/features/integrated/v1/schema.json`
- `reports/feature_integration.json`

</div>

**Next:** Notebook `11` (Model Training -- Baseline).